[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/04_activation_and_gating.ipynb)

# 04. Activation and gating

같은 작은 입력에서 비선형 함수와 gated FFN이 출력을 어떻게 바꾸는지 본다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. ReLU → LeakyReLU

가장 단순한 threshold 계열.


In [ ]:
x = torch.tensor([-3., -1., 0., 1., 3.], device=device)
print("ReLU      :", F.relu(x))
print("LeakyReLU :", F.leaky_relu(x, negative_slope=0.1))


In [ ]:
_ = profile_call("ReLU", F.relu, x)
_ = profile_call("LeakyReLU", lambda z: F.leaky_relu(z, 0.1), x)


## 2. GELU → SiLU

Transformer 계열에서 자주 쓰이는 부드러운 activation.


In [ ]:
print("GELU:", F.gelu(x))
print("SiLU:", F.silu(x))


In [ ]:
_ = profile_call("GELU", F.gelu, x)
_ = profile_call("SiLU", F.silu, x)


## 3. GLU

한 projection을 두 갈래로 나누고 한쪽을 gate로 사용한다.


In [ ]:
h = torch.tensor([[1., 2., 3., 4., -1., 0., 1., 2.]], device=device)
a, b = h.chunk(2, dim=-1)
y = a * torch.sigmoid(b)
print("a:", a)
print("gate:", torch.sigmoid(b))
print("GLU:", y)


In [ ]:
_ = profile_call("GLU explicit", lambda z: z.chunk(2, -1)[0] * torch.sigmoid(z.chunk(2, -1)[1]), h)


## 4. GEGLU → SwiGLU

gate activation만 바꿔 비교한다.


In [ ]:
u = torch.tensor([[1., -2., 0.5, 3.]], device=device)
v = torch.tensor([[0.5, 1., -1., 2.]], device=device)

geglu = u * F.gelu(v)
swiglu = u * F.silu(v)

print("GEGLU :", geglu)
print("SwiGLU:", swiglu)


In [ ]:
_ = profile_call("GEGLU", lambda p, q: p * F.gelu(q), u, v)
_ = profile_call("SwiGLU", lambda p, q: p * F.silu(q), u, v)


## References and provenance

**[4.1] GELU**
- 출처: Hendrycks & Gimpel, Gaussian Error Linear Units
- 이 노트북에서 가져온 부분: 부드러운 activation

**[4.2] GLU/GEGLU**
- 출처: Dauphin et al.; Shazeer, GLU Variants Improve Transformer
- 이 노트북에서 가져온 부분: gated FFN

**[4.3] SwiGLU**
- 출처: Shazeer; LLaMA and modern LLM implementations
- 이 노트북에서 가져온 부분: SiLU gate
